# 🧪 Experimento alternativo: Red Neuronal con Triplet Loss

Este notebook explora una variante del modelo siamés entrenada con función de pérdida triplet loss.  
Aunque se obtuvieron mejoras parciales en métricas top-k (NDCG, MAP), el orden global del ranking se volvió inestable.  
Por ello, este enfoque no se incluye en el capítulo principal de resultados del TFM, pero se documenta aquí como referencia.


## 1. Carga y Preparación de Datos

### Montaje de Google Drive

In [1]:
# ─── MONTAJE DE GOOGLE DRIVE ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Definición de rutas y mapping

In [2]:
# ─── IMPORTS ──────────────────────────────────────────────────────────────
import pandas as pd
import os
import numpy as np

# Ruta a la carpeta con los archivos CSV
BASE_PATH = "/content/drive/MyDrive/TFM/interim"

In [3]:
MAPPING = {
    'cdatos':    'Ciencia_de_datos',
    'ingdatos':  'Ingeniero_de_datos',
    'jurista':   'Jurista',
    'traductor': 'Traductor_de_inglés'
}

MAX_TOKENS = 300

### Carga de CVs y ofertas

In [4]:
def load_cvs(path, mapping):
    df = pd.read_csv(path)

    # Filtrar solo nombres que tengan uno de los códigos válidos
    mask = df['Nombre del archivo'].str.contains(r'_(cdatos|ingdatos|jurista|traductor)_')
    df_validos = df[mask].copy()
    df_invalidos = df[~mask]['Nombre del archivo'].tolist()

    if df_invalidos:
        print("Se excluyen estos nombres de archivo (no parecen CVs válidos):")
        print("\n".join(df_invalidos))

    # Ahora sí extraemos el código solo de los válidos
    df_validos['code'] = df_validos['Nombre del archivo'].str.extract(r'_(cdatos|ingdatos|jurista|traductor)_')[0]
    df_validos['category'] = df_validos['code'].map(mapping)

    # Si hay algo que no mapeó, lo advertimos
    unmapped = df_validos['category'].isna()
    if unmapped.any():
        bad = df_validos.loc[unmapped, 'Nombre del archivo'].tolist()
        raise ValueError("Estos CVs tienen un código válido pero no están en MAPPING:\n" +
                         "\n".join(bad))

    df_validos = df_validos.rename(columns={'Texto extraído': 'cv_text', 'Nombre del archivo': 'cv_id'})
    return df_validos[['cv_id', 'cv_text', 'category']]

def load_offers(offer_files):
    offers = []
    for cat, fp in offer_files.items():
        df = pd.read_csv(fp).rename(columns={'descripcion_oferta': 'offer_text'})
        df['offer_id'] = df.index.astype(str) + f'_{cat}'
        df['offer_category'] = cat
        offers.append(df[['offer_id', 'offer_text', 'offer_category']])
    return pd.concat(offers, ignore_index=True)

Se cargan los archivos CSV de currículums (CVs) y ofertas de empleo.

En el caso de los CVs:

- Se filtran solo los archivos cuyo nombre contiene un código de categoría válido

- Se verifica que todos los códigos extraídos tengan una correspondencia en el diccionario `MAPPING`.

- Se lanza un error si se detectan CVs con códigos válidos pero no definidos en el `MAPPING`.

En el caso de las ofertas, se construye un identificador único por oferta y se etiqueta cada una con su categoría correspondiente.

Esto garantiza que solo se trabajará con ejemplos consistentes y etiquetados correctamente.

In [5]:
# ─── CARGA DE DATOS ───────────────────────────────────────────────────────
# CVs de entrenamiento
cvs_path = os.path.join(BASE_PATH, "cvs_train_preprocesado.csv")
df_cvs = load_cvs(cvs_path, MAPPING)

# Ofertas por categoría
offer_files = {
    'Ciencia_de_datos': os.path.join(BASE_PATH, 'Ciencia_de_datos_España_ofertas.csv'),
    'Ingeniero_de_datos': os.path.join(BASE_PATH, 'Ingeniero_de_datos_España_ofertas.csv'),
    'Traductor_de_inglés': os.path.join(BASE_PATH, 'Traductor_de_inglés_Málaga_ofertas.csv'),
    'Jurista': os.path.join(BASE_PATH, 'Jurista_Málaga_ofertas.csv')
}

offers = load_offers(offer_files)


/tmp/ipython-input-4-1671012236.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['Nombre del archivo'].str.contains(r'_(cdatos|ingdatos|jurista|traductor)_')


Se excluyen estos nombres de archivo (no parecen CVs válidos):
CV_numeropdf_oferta_numerooferta.pdf


### Creación de pares CV-oferta

In [6]:
def make_pairs(df_cvs, df_offers):
    df_cvs = df_cvs.assign(key=1)
    df_offers = df_offers.assign(key=1)
    pairs = df_cvs.merge(df_offers, on='key').drop('key', axis=1)
    pairs['label'] = (pairs['category'] == pairs['offer_category']).astype(int)
    return pairs.rename(columns={'category':'cv_category'})

Se genera el conjunto de entrenamiento combinando todos los CVs con todas las ofertas mediante un producto cartesiano.

Cada par CV-oferta recibe una etiqueta binaria:
- `label = 1` si la categoría del CV coincide con la categoría de la oferta.
- `label = 0` en caso contrario.

Este etiquetado permite entrenar el modelo como una tarea de clasificación binaria, donde el objetivo es predecir si un par es un "match" (coincidente) o no.  
Se imprime además la distribución de clases para verificar el balance del dataset resultante.

In [7]:
pairs = make_pairs(df_cvs, offers)

### Verificación de la distribución

In [8]:
# ─── VERIFICACIÓN FINAL ──────────────────────────────────────────────────
print("CVs cargados:", len(df_cvs))
print("Ofertas cargadas:", len(offers))
print("Total de pares generados:", len(pairs))
print("Distribución de clases (label):")
print(pairs['label'].value_counts())


CVs cargados: 72
Ofertas cargadas: 19
Total de pares generados: 1368
Distribución de clases (label):
label
0    1024
1     344
Name: count, dtype: int64


## 2. División del Dataset y Preprocesamiento

### Split de entrenamiento y validación

Primero se divide el dataset en entrenamiento y validación, asegurando que no haya CVs duplicados entre ambos conjuntos.  
La división se hace a nivel de `cv_id`, manteniendo el 80% para entrenamiento y el 20% para validación.

Después, se realiza una limpieza básica del texto tanto en los CVs como en las ofertas:
- Se eliminan tildes y signos de puntuación.
- Se convierte todo a minúsculas.
- Se normalizan los espacios.

Esta limpieza estandariza el texto y reduce ruido para mejorar la calidad del entrenamiento posterior.

In [9]:
# ─── SPLIT DE ENTRENAMIENTO Y VALIDACIÓN ─────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, ndcg_score

# Dividimos el dataset de pares con labels (80% train, 20% valid)
unique_cvs = pairs['cv_id'].unique()
cv_train_ids, cv_val_ids = train_test_split(unique_cvs, test_size=0.2, random_state=42)

pairs_train = pairs[pairs['cv_id'].isin(cv_train_ids)].copy()
pairs_val = pairs[pairs['cv_id'].isin(cv_val_ids)].copy()

print(f"Pares para entrenamiento: {len(pairs_train)}")
print(f"Pares para validación: {len(pairs_val)}")
print("Distribución en train:")
print(pairs_train['label'].value_counts())
print("Distribución en validación:")
print(pairs_val['label'].value_counts())


Pares para entrenamiento: 1083
Pares para validación: 285
Distribución en train:
label
0    812
1    271
Name: count, dtype: int64
Distribución en validación:
label
0    212
1     73
Name: count, dtype: int64


### Limpieza de texto

In [10]:
def limpiar_texto(texto):
    # Eliminar tildes y normalizar unicode
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8', 'ignore')

    # Pasar a minúsculas
    texto = texto.lower()

    # Sustituir signos de puntuación por espacios
    texto = re.sub(r"[\.,;:!\?()\[\]\"']", " ", texto)

    # Sustituir múltiples espacios por uno
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

In [11]:
# ─── IMPORTACIONES ────────────────────────────────────────────────────────
import re
import unicodedata
from tqdm import tqdm

tqdm.pandas()


In [12]:
# ─── APLICAMOS LIMPIEZA AL DATASET DE PARES ──────────────────────────────
for campo in ['cv_text', 'offer_text']:
    print(f"Limpiando campo: {campo}")
    pairs_train[campo] = pairs_train[campo].progress_apply(limpiar_texto)
    pairs_val[campo] = pairs_val[campo].progress_apply(limpiar_texto)


Limpiando campo: cv_text


100%|██████████| 285/285 [00:00<00:00, 7358.70it/s]


Limpiando campo: offer_text


100%|██████████| 285/285 [00:00<00:00, 6916.65it/s]


### Truncado y augmentación

In [13]:
def truncar_texto(texto, max_tokens=MAX_TOKENS):
    tokens = texto.split()
    if len(tokens) > max_tokens:
        return " ".join(tokens[:max_tokens])
    else:
        return texto

def permutar_frases(texto):
    import random, re
    frases = [f.strip() for f in re.split(r"[\.!?]", texto) if f.strip()]
    if len(frases) <= 1:
        return texto  # no permutamos si no hay más de una frase
    random.shuffle(frases)
    return ". ".join(frases) + "."

def augment_train(pairs):
    pos = pairs[pairs.label==1].copy()
    pos['cv_text'] = pos['cv_text'].apply(permutar_frases)
    pos['offer_text'] = pos['offer_text'].apply(permutar_frases)
    pos['cv_id']    = pos['cv_id'] + "_aug"
    pos['offer_id'] = pos['offer_id'] + "_aug"
    return pd.concat([pairs, pos], ignore_index=True)

Primero se aplican dos transformaciones clave sobre los textos:

1. **Truncado de texto**: se limitan los CVs y ofertas a un máximo de `MAX_TOKENS` palabras. Esto evita entradas excesivamente largas que podrían desbordar la memoria durante el entrenamiento.

2. **Aumento de datos (data augmentation)**: se duplican los pares positivos (`label = 1`) generando una nueva versión donde se permutan aleatoriamente las frases del CV y de la oferta.  
   Esto introduce variabilidad en el orden del contenido sin alterar su significado, lo que ayuda a mejorar la robustez del modelo.

Los nuevos ejemplos se identifican con sufijos `_aug` en sus `cv_id` y `offer_id`.

In [14]:
# ─── TRUNCADO DE TEXTOS LARGOS ─────────────────────────────────

for campo in ['cv_text', 'offer_text']:
    pairs_train[campo] = pairs_train[campo].apply(truncar_texto)
    pairs_val[campo] = pairs_val[campo].apply(truncar_texto)


In [15]:
# ─── AUGMENT TRAIN ─────────────────────────────────────────────────────────
pairs_train = augment_train(pairs_train)

print(f"Nuevo tamaño de pairs_train tras augmentación: {len(pairs_train)}")
print("Distribución tras augmentación:")
print(pairs_train['label'].value_counts())


Nuevo tamaño de pairs_train tras augmentación: 1354
Distribución tras augmentación:
label
0    812
1    542
Name: count, dtype: int64


### Guardado de pares limpios

In [16]:
# ─── GUARDAR COPIAS PREPROCESADAS ──────────────────────────────
pairs_train.to_csv(os.path.join(BASE_PATH, "pairs_train_clean.csv"), index=False)
pairs_val.to_csv(os.path.join(BASE_PATH, "pairs_val_clean.csv"), index=False)


### Tokenización

In [17]:
from collections import Counter
def construir_vocab(textos, vocab_size=8000, min_freq=2):
    c = Counter()
    for t in textos: c.update(t.split())
    vocab = {'<PAD>':0,'<UNK>':1}
    idx = 2
    for tok,f in c.most_common():
        if f < min_freq or len(vocab)>=vocab_size: break
        vocab[tok] = idx; idx+=1
    return vocab

def text_to_indices(texto, vocab, max_len=300):
    tokens = texto.split()[:max_len]
    ids = [vocab.get(tok, vocab['<UNK>']) for tok in tokens]
    ids += [vocab['<PAD>']] * (max_len-len(ids))
    return ids

def tokenize_pairs(pairs, vocab):
    df = pairs.copy()
    df['cv_input_ids']    = df['cv_text'].apply(lambda t: text_to_indices(t, vocab))
    df['offer_input_ids'] = df['offer_text'].apply(lambda t: text_to_indices(t, vocab))
    return df

Se construye el vocabulario del modelo a partir de todos los textos de CVs y ofertas (entrenamiento y validación), limitando el tamaño máximo del vocabulario y filtrando por frecuencia mínima.

Con ese vocabulario, se tokenizan los textos:
- Cada palabra se convierte en su índice correspondiente.
- Se aplica padding o truncado para que todos los ejemplos tengan la misma longitud.

El resultado es un dataset preparado para alimentar directamente a una red neuronal.

In [18]:
# ─── 1. CONSTRUIR EL VOCABULARIO SOBRE TODOS LOS TEXTOS ─────────────────────
# Concatenamos todos los textos de CVs y ofertas (train y val)
todos_los_textos = pd.concat([
    pairs_train['cv_text'],
    pairs_train['offer_text'],
    pairs_val['cv_text'],
    pairs_val['offer_text']
])

# Construimos vocabulario
vocab = construir_vocab(todos_los_textos, vocab_size=8000, min_freq=2)

print(f"Vocabulario creado con {len(vocab)} tokens.")


Vocabulario creado con 2631 tokens.


In [19]:
pairs_train_tokenized = tokenize_pairs(pairs_train, vocab)
pairs_val_tokenized = tokenize_pairs(pairs_val, vocab)

print(f"Pairs train tokenized shape: {pairs_train_tokenized.shape}")
print(f"Pairs val tokenized shape: {pairs_val_tokenized.shape}")


Pairs train tokenized shape: (1354, 9)
Pairs val tokenized shape: (285, 9)


✅ La tokenización se ha aplicado correctamente: cada par CV–oferta tiene ahora sus representaciones en índices (`cv_input_ids` y `offer_input_ids`), listas para ser usadas en el modelo.

Ambos datasets (`train` y `val`) contienen 9 columnas, lo que indica que conservan los campos originales más los tokens generados.


## 3. Arquitectura del Modelo

### Definición del encoder CNN-LSTM

**SiameseEncoderCNNLSTM**:  
   Este encoder transforma una secuencia de tokens en una representación vectorial.  
   Su estructura combina:
   - Una capa de embeddings.
   - Una convolución 1D para extraer patrones locales.
   - Una LSTM bidireccional que captura dependencias a largo plazo.
   - Un max-pooling global para obtener una representación fija del texto.

In [20]:
# Import torch
import torch
import torch.nn as nn
import torch.nn.functional as F

class SiameseEncoderCNNLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, conv_out=64,
                 lstm_hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.conv      = nn.Conv1d(emb_dim, conv_out, kernel_size=5, padding=2)
        self.lstm      = nn.LSTM(conv_out, lstm_hidden,
                                 num_layers=1, batch_first=True,
                                 bidirectional=True)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x):                          # (B,L)
        x = self.embedding(x).transpose(1, 2)      # (B,E,L)
        x = F.relu(self.conv(x)).transpose(1, 2)    # (B,L,C)
        o, _ = self.lstm(x)                         # (B,L,2H)
        o_max, _ = o.max(dim=1)                    # (B,2H)
        return self.dropout(o_max)

### Modelo Siamese con MLP final

In [21]:
class SiameseSimilarityModelEnhanced(nn.Module):
    """Devuelve un score real; NO lleva sigmoide, usamos MarginRankingLoss."""
    def __init__(self, vocab_size, encoder_cls, encoder_kwargs, seq_len=300):
        super().__init__()
        self.encoder = encoder_cls(vocab_size, **encoder_kwargs)
        with torch.no_grad():
            dummy = torch.zeros(1, seq_len, dtype=torch.long)
            d_enc = self.encoder(dummy).shape[1]

        self.mlp = nn.Sequential(
            nn.Linear(2 * d_enc, d_enc),
            nn.ReLU(),
            nn.Dropout(encoder_kwargs.get("dropout", .3)),
            nn.Linear(d_enc, 1),
            nn.Identity()               # salida escalar libre
        )

    def forward(self, a, b):
        return self.mlp(torch.cat([self.encoder(a), self.encoder(b)], 1)).squeeze(1)

### Hiperparámetros y construcción del modelo

Se definen los hiperparámetros clave del encoder: dimensión de embeddings, tamaño de los filtros convolucionales, dimensión oculta del LSTM y tasa de dropout.

A continuación, se instancia el modelo `SiameseSimilarityModelEnhanced`, indicando la clase de encoder y los parámetros definidos.  
El modelo se mueve a GPU si está disponible (`"cuda"`).

In [22]:
VOCAB_SIZE = len(vocab)  # el vocabulario que creaste en Fase 1

# Definición del encoder “mejorado”
encoder_kwargs = {
    'emb_dim':      128,
    'conv_out':     64,
    'lstm_hidden':  128,
    'dropout':      0.3
}

# Creas el modelo pasándole la clase del encoder y sus argumentos
model = SiameseSimilarityModelEnhanced(
    vocab_size   = VOCAB_SIZE,
    encoder_cls  = SiameseEncoderCNNLSTM,
    encoder_kwargs = encoder_kwargs
).to("cuda")



### Pérdida y optimizador

In [23]:
# Creamos la función de pérdida
from torch import nn

loss_fn = nn.MarginRankingLoss(margin=1.0)


In [24]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)


## 4. Entrenamiento del Modelo

### Dataset y DataLoaders

In [25]:
# -------------------------------------------------------------------
# DATASETS -----------------------------------------------------------
# -------------------------------------------------------------------
class TripletDataset(torch.utils.data.Dataset):
    """
    Devuelve tripletas (cv, oferta_pos, oferta_neg).  Pensado para
    MarginRankingLoss (score_pos > score_neg).
    """
    def __init__(self, df_pairs, n_negatives: int = 1, shuffle_offers: bool = True):
        """
        df_pairs: DataFrame con columnas:
          - cv_id , cv_input_ids
          - offer_id , offer_input_ids
          - label  (1 = match, 0 = no-match)
        """
        super().__init__()
        self.n_neg = n_negatives
        self.shuffle_offers = shuffle_offers

        # separamos positivas y negativas por cv
        pos = df_pairs[df_pairs.label == 1]
        neg = df_pairs[df_pairs.label == 0]

        self.triplets = []
        # Por cada CV, seleccionamos una o más ofertas positivas y varias negativas para formar tripletas (anchor, pos, neg).
        for cv_id, pos_grp in pos.groupby("cv_id"):
            neg_grp = neg[neg.cv_id == cv_id]
            if neg_grp.empty:                # saltamos CV sin negativas
                continue
            for _idx, pos_row in pos_grp.iterrows():
                # elegimos n_neg negativas distintas
                neg_rows = neg_grp.sample(
                    n=min(self.n_neg, len(neg_grp)),
                    replace=False,
                    random_state=None
                )
                for __, neg_row in neg_rows.iterrows():
                    self.triplets.append(
                        (
                            pos_row.cv_input_ids,
                            pos_row.offer_input_ids,
                            neg_row.offer_input_ids,
                            cv_id,                             # para logging opcional
                        )
                    )

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        anchor, pos, neg, cv_id = self.triplets[idx]
        return {
            "anchor": torch.tensor(anchor, dtype=torch.long),
            "pos":    torch.tensor(pos,    dtype=torch.long),
            "neg":    torch.tensor(neg,    dtype=torch.long),
            "cv_id":  cv_id,
        }


class PairsDataset(torch.utils.data.Dataset):
    """Pares (cv, oferta, label) — idéntico al que ya tenías, pero con cv_id dentro."""
    def __init__(self, df):
        self.cv_ids     = df.cv_id.tolist()
        self.cv_inputs  = df.cv_input_ids.tolist()
        self.of_inputs  = df.offer_input_ids.tolist()
        self.labels     = df.label.tolist()

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        return {
            "cv_input_ids":    torch.tensor(self.cv_inputs[i], dtype=torch.long),
            "offer_input_ids": torch.tensor(self.of_inputs[i], dtype=torch.long),
            "label":           torch.tensor(self.labels[i],   dtype=torch.float),
            "cv_id":           self.cv_ids[i],
        }

In [26]:
display(pairs_train_tokenized.head(15))

,cv_id,cv_text,cv_category,offer_id,offer_text,offer_category,label,cv_input_ids,offer_input_ids
0,CV_01_cdatos_02.pdf,curriculum vitae nombre laura fernandez ramire...,Ciencia_de_datos,0_Ciencia_de_datos,acerca del empleo los ingenieros de software j...,Ciencia_de_datos,1,"[63, 64, 81, 212, 456, 1590, 134, 212, 1612, 4...","[57, 12, 54, 16, 114, 2, 89, 107, 515, 152, 51..."
1,CV_01_cdatos_02.pdf,curriculum vitae nombre laura fernandez ramire...,Ciencia_de_datos,1_Ciencia_de_datos,acerca del empleo unete a inetum como responsa...,Ciencia_de_datos,1,"[63, 64, 81, 212, 456, 1590, 134, 212, 1612, 4...","[57, 12, 54, 549, 10, 1170, 27, 223, 2, 24, 77..."
2,CV_01_cdatos_02.pdf,curriculum vitae nombre laura fernandez ramire...,Ciencia_de_datos,2_Ciencia_de_datos,acerca del empleo estamos ampliando nuestro eq...,Ciencia_de_datos,1,"[63, 64, 81, 212, 456, 1590, 134, 212, 1612, 4...","[57, 12, 54, 151, 1187, 58, 20, 2, 496, 2, 8, ..."
3,CV_01_cdatos_02.pdf,curriculum vitae nombre laura fernandez ramire...,Ciencia_de_datos,3_Ciencia_de_datos,acerca del empleo seguimos buscando talento y ...,Ciencia_de_datos,1,"[63, 64, 81, 212, 456, 1590, 134, 212, 1612, 4...","[57, 12, 54, 1219, 325, 1220, 3, 140, 423, 14,..."
4,CV_01_cdatos_02.pdf,curriculum vitae nombre laura fernandez ramire...,Ciencia_de_datos,4_Ciencia_de_datos,acerca del empleo buscamos un/a consultor/a de...,Ciencia_de_datos,1,"[63, 64, 81, 212, 456, 1590, 134, 212, 1612, 4...","[57, 12, 54, 105, 229, 752, 2, 94, 145, 5, 111..."
5,CV_01_cdatos_02.pdf,curriculum vitae nombre laura fernandez ramire...,Ciencia_de_datos,0_Ingeniero_de_datos,acerca del empleo los ingenieros de software j...,Ingeniero_de_datos,0,"[63, 64, 81, 212, 456, 1590, 134, 212, 1612, 4...","[57, 12, 54, 16, 114, 2, 89, 107, 515, 152, 51..."
6,CV_01_cdatos_02.pdf,curriculum vitae nombre laura fernandez ramire...,Ciencia_de_datos,1_Ingeniero_de_datos,acerca del empleo los ingenieros de software j...,Ingeniero_de_datos,0,"[63, 64, 81, 212, 456, 1590, 134, 212, 1612, 4...","[57, 12, 54, 16, 114, 2, 89, 107, 515, 152, 51..."
7,CV_01_cdatos_02.pdf,curriculum vitae nombre laura fernandez ramire...,Ciencia_de_datos,2_Ingeniero_de_datos,acerca del empleo hola somos lis data solution...,Ingeniero_de_datos,0,"[63, 64, 81, 212, 456, 1590, 134, 212, 1612, 4...","[57, 12, 54, 1072, 385, 467, 24, 172, 34, 65, ..."
8,CV_01_cdatos_02.pdf,curriculum vitae nombre laura fernandez ramire...,Ciencia_de_datos,3_Ingeniero_de_datos,acerca del empleo te entusiasta el analisis de...,Ingeniero_de_datos,0,"[63, 64, 81, 212, 456, 1590, 134, 212, 1612, 4...","[57, 12, 54, 141, 856, 13, 28, 2, 8, 3, 527, 2..."
9,CV_01_cdatos_02.pdf,curriculum vitae nombre laura fernandez ramire...,Ciencia_de_datos,4_Ingeniero_de_datos,acerca del empleo role splunk software enginee...,Ingeniero_de_datos,0,"[63, 64, 81, 212, 456, 1590, 134, 212, 1612, 4...","[57, 12, 54, 1128, 233, 89, 234, 1129, 935, 11..."


In [27]:
from torch.utils.data import DataLoader

# ---------- ENTRENAMIENTO ----------
# Devuelve (cv_ids , pos_offer_ids , neg_offer_ids)
train_dataset = TripletDataset(
    pairs_train_tokenized,          # DataFrame con ofertas etiquetadas
    n_negatives = 14,         # nº de ofertas negativas por CV
    shuffle_offers = True    # barajamos las negativas cada época
)

# ‣ Lote = nº de tripletas; 32 suele ir bien en GPU medianas
train_loader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle    = True,
    drop_last  = True        # asegura batches completos
)

# ---------- VALIDACIÓN -------------
# Seguimos usando pares para calcular NDCG/metrics
val_dataset = PairsDataset(pairs_val_tokenized)

# ‣ 19 ofertas por CV → usa múltiplos de 19
val_loader = DataLoader(
    val_dataset,
    batch_size = 19,         # 1 CV completo por lote
    shuffle    = False
)


In [28]:
from sklearn.metrics import ndcg_score   ### NEW

def compute_ndcg_at_k(y_true, y_pred, cv_ids, k: int = 4) -> float:
    """NDCG@k medio agrupado por cv_id."""
    df = pd.DataFrame({"cv_id": cv_ids, "y_true": y_true, "y_pred": y_pred})
    ndcgs = []
    for _, grp in df.groupby("cv_id"):
        # asegura orden descendente de predicción
        grp = grp.sort_values("y_pred", ascending=False)
        if grp.y_true.sum() == 0:            # sin positivos → ignoramos
            continue
        ndcgs.append(
            ndcg_score(grp.y_true.values.reshape(1, -1),
                       grp.y_pred.values.reshape(1, -1),
                       k=k)
        )
    return float(np.mean(ndcgs)) if ndcgs else 0.0

### Funciones de entrenamiento y evaluación

In [29]:
def train_model(
    model,
    pairs_train_df,
    pairs_val_df,
    device="cuda",
    epochs: int = 20,
    patience: int = 4,
    show_every: int = 2,
    margin: float = 0.2,
    save_path: str = "best_model.pt",
):
    """
    * Entrena con tripletas + MarginRankingLoss.
    * Valida con pares + NDCG@4.
    """

    # ---------- DataLoaders ----------
    train_loader = DataLoader(
        TripletDataset(pairs_train_df, n_negatives=1, shuffle_offers=True),
        batch_size=32,
        shuffle=True,
        drop_last=True,                       # batches completos = estable
    )

    val_loader = DataLoader(
        PairsDataset(pairs_val_df),
        batch_size=19,                        # 19 ofertas x CV  → 1 CV por batch
        shuffle=False,
    )

    # ---------- Loss + Optimizador ----
    loss_fn   = nn.MarginRankingLoss(margin=margin)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

    best_ndcg = 0.0
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        # ======== TRAIN ==============
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            anc = batch["anchor"].to(device)
            pos = batch["pos"].to(device)
            neg = batch["neg"].to(device)

            optimizer.zero_grad()
            s_pos = model(anc, pos)          # score(anchor,pos)
            s_neg = model(anc, neg)          # score(anchor,neg)

            # target = 1 → queremos s_pos > s_neg + margin
            triplet_y = torch.ones_like(s_pos)
            loss      = loss_fn(s_pos, s_neg, triplet_y)

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # ======== VALID ==============
        model.eval()
        y_true, y_pred, cv_ids = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                cv  = batch["cv_input_ids"].to(device)
                off = batch["offer_input_ids"].to(device)
                lbl = batch["label"].to(device)

                scores = model(cv, off)

                y_true.extend(lbl.cpu().numpy())
                y_pred.extend(scores.cpu().numpy())
                cv_ids.extend(batch["cv_id"])

        ndcg4 = compute_ndcg_at_k(np.array(y_true), np.array(y_pred), cv_ids, k=4)

        # ----- Early-stopping --------
        if ndcg4 > best_ndcg:
            best_ndcg = ndcg4
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
        else:
            epochs_no_improve += 1

        # ----- LOG -------------------
        if epoch % show_every == 0 or epoch == epochs:
            print(
                f"Epoch {epoch:02d}/{epochs} | "
                f"TrainLoss {train_loss:.4f} | "
                f"NDCG@4 {ndcg4:.4f} | Best {best_ndcg:.4f}"
            )

        if epochs_no_improve >= patience:
            print(f"Early-stopping (no mejora en {patience} épocas)")
            break

    # --------- Fin --------------
    model.load_state_dict(torch.load(save_path))
    print(f"\nEntrenamiento finalizado. Mejor NDCG@4 = {best_ndcg:.4f}")
    return model

### Ejecución del entrenamiento

Se lanza el entrenamiento del modelo utilizando la función `train_model`.

- Se detecta automáticamente si hay GPU disponible (`"cuda"`).
- Se especifican hiperparámetros clave como:
  - `epochs`: número máximo de épocas de entrenamiento.
  - `patience`: número de épocas sin mejora tras las cuales se activa el early stopping.
  - `threshold`: umbral para considerar un score como positivo en las métricas.
  - `show_every`: frecuencia con la que se imprime el estado de entrenamiento.
  - `save_path`: ruta donde se guarda el mejor modelo encontrado.

El modelo devuelto ya contiene los pesos entrenados óptimos según el F1 en validación.

In [30]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = train_model(
    model,
    pairs_train_df   = pairs_train_tokenized,
    pairs_val_df     = pairs_val_tokenized,
    device           = device,
    epochs           = 250,
    patience         = 75,
    show_every       = 7,
    margin           = 0.2,
    save_path        = "mejor_modelo.pt"
)


Epoch 07/250 | TrainLoss 0.1950 | NDCG@4 0.2000 | Best 0.3041
Epoch 14/250 | TrainLoss 0.1831 | NDCG@4 0.2521 | Best 0.3041
Epoch 21/250 | TrainLoss 0.1862 | NDCG@4 0.2148 | Best 0.3041
Epoch 28/250 | TrainLoss 0.0252 | NDCG@4 0.6424 | Best 0.7169
Epoch 35/250 | TrainLoss 0.0042 | NDCG@4 0.7594 | Best 0.7594
Epoch 42/250 | TrainLoss 0.0061 | NDCG@4 0.7239 | Best 0.8146
Epoch 49/250 | TrainLoss 0.0010 | NDCG@4 0.7463 | Best 0.8146
Epoch 56/250 | TrainLoss 0.0003 | NDCG@4 0.7333 | Best 0.8146
Epoch 63/250 | TrainLoss 0.0008 | NDCG@4 0.6943 | Best 0.8146
Epoch 70/250 | TrainLoss 0.0002 | NDCG@4 0.7724 | Best 0.8146
Epoch 77/250 | TrainLoss 0.0004 | NDCG@4 0.7888 | Best 0.8146
Epoch 84/250 | TrainLoss 0.0015 | NDCG@4 0.7125 | Best 0.8146
Epoch 91/250 | TrainLoss 0.0000 | NDCG@4 0.7576 | Best 0.8146
Epoch 98/250 | TrainLoss 0.0026 | NDCG@4 0.7740 | Best 0.8146
Epoch 105/250 | TrainLoss 0.0000 | NDCG@4 0.7628 | Best 0.8146
Early-stopping (no mejora en 75 épocas)

Entrenamiento finalizado. Me

## 5. Evaluación y Calibración

### Evaluación del modelo entrenado

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 1) EVALUATION (scores crudos por defecto) ───────────────────────────────────
# ──────────────────────────────────────────────────────────────────────────────
def evaluate(model, loader, device: str = "cuda", apply_sigmoid: bool = False):
    """
    Evalúa el modelo sobre un DataLoader y devuelve:
      y_true : array etiquetas {0,1}
      y_pred : array scores (crudos o sigmoide)

    Si `apply_sigmoid=True`, convierte los scores con σ(x)=1/(1+e^{-x}).
    """
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            cv  = batch["cv_input_ids"].to(device)
            off = batch["offer_input_ids"].to(device)
            lbl = batch["label"].to(device)

            scores = model(cv, off)                 # ← ahora rangos reales ℝ
            if apply_sigmoid:
                scores = torch.sigmoid(scores)      # [0,1]

            y_true.extend(lbl.cpu().numpy())
            y_pred.extend(scores.cpu().numpy())

    return np.array(y_true), np.array(y_pred)

## 6. Inferencia en CVs de Test

### Carga y limpieza de los CVs de test

In [31]:
import os, re, unicodedata, joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# ------------------------------------------------------------------------
# 6.1  Carga del fichero de test y pre-procesado idéntico al de train
# ------------------------------------------------------------------------
TEST_CSV = os.path.join(BASE_PATH, "cvs_test_preprocesado.csv")

df_test = (
    pd.read_csv(TEST_CSV)
      .rename(columns={"Texto extraído": "cv_text",
                       "Nombre del archivo": "cv_id"})
)

print("CVs de test cargados:", len(df_test))

# ─ limpieza y truncado ──────────────────────────────────────────────────
df_test["cv_text"] = (
    df_test["cv_text"]
      .apply(limpiar_texto)
      .apply(truncar_texto)                      # usa MAX_TOKENS global
)

CVs de test cargados: 13


Se realiza el preprocesado completo para aplicar el modelo a nuevos CVs sin etiqueta:

1. **Carga de test**: se lee el archivo `cvs_test_preprocesado.csv`, renombrando las columnas relevantes.

### Combinación CV-oferta y Tokenización

In [32]:
# ------------------------------------------------------------------------
# 6.2  Tokenización de CVs y Ofertas (ya cargadas)
# ------------------------------------------------------------------------
df_test["cv_input_ids"]  = df_test["cv_text"].apply(
    lambda x: text_to_indices(x, vocab, max_len=MAX_TOKENS)
)

df_ofertas = offers.copy()
df_ofertas["offer_text"] = (
    df_ofertas["offer_text"]
      .apply(limpiar_texto)
      .apply(truncar_texto)
)
df_ofertas["offer_input_ids"] = df_ofertas["offer_text"].apply(
    lambda x: text_to_indices(x, vocab, max_len=MAX_TOKENS)
)

print("Ofertas en inferencia:", len(df_ofertas))

Ofertas en inferencia: 19


2. **Limpieza y truncado**: se aplica la misma limpieza y limitación de longitud usada en entrenamiento, garantizando consistencia.

3. **Tokenización**: los textos de los CVs y las ofertas se convierten a secuencias de índices utilizando el vocabulario previamente construido.

4. **Producto cartesiano**: se generan todos los pares posibles `CV_test × Oferta`, creando el conjunto sobre el que se hará inferencia.

In [33]:
# ------------------------------------------------------------------------
# 6.3  Producto cartesiano CV_test × Ofertas
# ------------------------------------------------------------------------
pairs_test = (
    df_test.assign(key=1)
           .merge(df_ofertas.assign(key=1), on="key")
           .drop("key", axis=1)
)
print("Total de pares a puntuar:", len(pairs_test))

Total de pares a puntuar: 247


### Tensores

5. **Conversión a tensores**: las secuencias tokenizadas se convierten a tensores PyTorch para poder pasarlas al modelo en batch.

Este bloque deja los datos completamente preparados para ejecutar el modelo y obtener los scores de adecuación.

In [34]:
# ─── CONVERSIÓN A TENSORES ─────────────────────────────────────────────────
cv_tensor     = torch.tensor(list(pairs_test['cv_input_ids']), dtype=torch.long)
offer_tensor  = torch.tensor(list(pairs_test['offer_input_ids']), dtype=torch.long)

In [35]:
# ------------------------------------------------------------------------
# 6.4  Dataset & DataLoader SOLO para inferencia (sin label)
# ------------------------------------------------------------------------
class InferenceDataset(Dataset):
    def __init__(self, df):
        self.cv_ids     = df.cv_id.tolist()
        self.offer_ids  = df.offer_id.tolist()
        self.cv_inputs  = df.cv_input_ids.tolist()
        self.off_inputs = df.offer_input_ids.tolist()

    def __len__(self): return len(self.cv_ids)

    def __getitem__(self, i):
        return {
            "cv_id":           self.cv_ids[i],
            "offer_id":        self.offer_ids[i],
            "cv_input_ids":    torch.tensor(self.cv_inputs[i],  dtype=torch.long),
            "offer_input_ids": torch.tensor(self.off_inputs[i], dtype=torch.long),
        }

infer_loader = DataLoader(
    InferenceDataset(pairs_test),
    batch_size=64,           # ⇒ ajusta a tu GPU
    shuffle=False,
)

### Predicción de scores calibrados

In [37]:
# ─── 2. Cálculo de scores SIN calibrador ──────────────────────────────────
with torch.no_grad():
    cv_tensor    = cv_tensor.to("cuda")       # <-- definidos en tu flujo
    offer_tensor = offer_tensor.to("cuda")

    logits = model(cv_tensor, offer_tensor).cpu()   # ℝ

    # Opción A: usar logits tal cual (para ranking)
    raw_scores = logits.numpy()

    # Opción B: mapear a (0,1) con sigmoide
    prob_scores = torch.sigmoid(logits).numpy()

In [38]:
 # ─── 3. Construcción del DataFrame final ──────────────────────────────────
df_resultados = pairs_test[['cv_id', 'offer_id', 'offer_text']].copy()

# Escoge la columna que prefieras:
df_resultados['score'] = prob_scores        # 0-1   (o usa raw_scores)

In [39]:
display(df_resultados)

,cv_id,offer_id,offer_text,score
0,cv00.pdf,0_Ciencia_de_datos,acerca del empleo los ingenieros de software j...,0.507745
1,cv00.pdf,1_Ciencia_de_datos,acerca del empleo unete a inetum como responsa...,0.476908
2,cv00.pdf,2_Ciencia_de_datos,acerca del empleo estamos ampliando nuestro eq...,0.490426
3,cv00.pdf,3_Ciencia_de_datos,acerca del empleo seguimos buscando talento y ...,0.496229
4,cv00.pdf,4_Ciencia_de_datos,acerca del empleo buscamos un/a consultor/a de...,0.501566
...,...,...,...,...
242,cv12.pdf,0_Jurista,acerca del empleo consejos haz un resumen del ...,0.339100
243,cv12.pdf,1_Jurista,acerca del empleo en ey tendras la oportunidad...,0.281071
244,cv12.pdf,2_Jurista,acerca del empleo es tu oportunidad ecointegra...,0.345782
245,cv12.pdf,3_Jurista,acerca del empleo descripcion del puesto y res...,0.305792


### Exportación de resultados

Finalmente, se exporta el ranking completo a un archivo CSV (`rankings_test_generado.csv`) que puede usarse para análisis o recomendaciones posteriores.

In [40]:
# ─── EXPORTACIÓN A CSV ─────────────────────────────────────────────────────
output_path = "/content/drive/MyDrive/TFM/red_neuronal/rankings_test_generado.csv"
df_resultados.to_csv(output_path, index=False)

print("✅ CSV generado con éxito:", output_path)

✅ CSV generado con éxito: /content/drive/MyDrive/TFM/red_neuronal/rankings_test_generado.csv
